In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Pick the first addable medicine and read its BEFORE stock in PCs.
    # Stock line looks like "\u2261 12 Strips / 120 PCs" (verified in CatalogTable.jsx + utils/units.js).
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    add_buttons = [b for b in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add_buttons, "No addable medicine found in the catalog."
    first_row = add_buttons[0].find_element(By.XPATH, "./ancestor::tr[1]")
    med_name = first_row.text.split("\n")[0]
    eq_text = first_row.find_element(By.XPATH, ".//*[contains(text(), '\u2261')]").text
    before_stock = int(re.findall(r"([\d,]+)\s+PCs?", eq_text)[-1].replace(",", ""))
    print("Medicine:", med_name)
    print("Before Stock:", before_stock)

    # Add 1 unit and complete the sale
    add_buttons[0].click()
    time.sleep(2)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Complete Sale') and not(contains(., 'Reviewed'))]"))).click()
    time.sleep(3)
    approve = [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Reviewed')]") if b.is_displayed()]
    if approve:
        print("Approval modal appeared, confirming...")
        approve[0].click()
        time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sale Completed']")))
    driver.find_element(By.XPATH, "//button[contains(., 'Start New Sale')]").click()
    time.sleep(2)

    # Open Medicines & Inventory (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Medicines & Inventory')]"))).click()
    search = wait.until(EC.visibility_of_element_located((By.XPATH, "//input[@aria-label='Search medicine or brand']")))
    search.clear()
    search.send_keys(med_name)
    time.sleep(2)

    # Open the same medicine's detail drawer (row click opens it, verified in InventoryTable.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "(//tr[contains(@class, 'med-row-tr')])[1]"))).click()
    drawer = wait.until(EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'med-drawer-backdrop')]")))
    eq_text2 = drawer.find_element(By.XPATH, ".//*[contains(text(), '\u2261')]").text
    after_stock = int(re.findall(r"([\d,]+)\s+PCs?", eq_text2)[-1].replace(",", ""))
    print("After Stock:", after_stock)

    assert after_stock < before_stock, f"Stock did not decrease (before={before_stock}, after={after_stock})."
    print("Current URL:", driver.current_url)
    print("PASS: Stock Update")
except Exception as e:
    print("FAIL: Stock Update")
    print("Error:", e)
    driver.save_screenshot("19_stock_update_FAIL.png")

In [ ]:
driver.quit()